[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/02_Introduction_to_ONNX/02_Why_ONNX_Matters/Why_ONNX_Matters_Apply.ipynb)

# 1.2 Why ONNX Matters — Apply

## Table of Contents
1. [Setup](#section-1)
2. [Cross-Framework Model Deployment](#section-2)
3. [Performance Comparison: Multiple Providers](#section-3)
4. [Multi-Target Deployment from Single Model](#section-4)
5. [Graph Optimization Effects](#section-5)
6. [Model Size and Memory Analysis](#section-6)
7. [Batch Processing and Throughput](#section-7)
8. [Building a Production Inference Pipeline](#section-8)
9. [Exercises](#section-9)

In [ ]:
!pip install onnx onnxruntime numpy matplotlib torch torchvision -q

<a id='section-1'></a>
## Section 1: Setup

In this notebook, we demonstrate the practical value of ONNX through hands-on experiments comparing framework execution with ONNX Runtime, measuring optimization effects, and building deployment pipelines.

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np
import torch
import torch.nn as nn
import time
import os
import matplotlib.pyplot as plt

print(f"ONNX: {onnx.__version__}")
print(f"ORT:  {ort.__version__}")
print(f"Torch: {torch.__version__}")
print(f"Providers: {ort.get_available_providers()}")

<a id='section-2'></a>
## Section 2: Cross-Framework Model Deployment

We demonstrate the core value proposition: train in PyTorch, deploy with ONNX Runtime. We build a CNN classifier and verify numerical equivalence after export.

The mathematical guarantee we verify:

$$\|f_{\text{PyTorch}}(x) - f_{\text{ORT}}(x)\|_\infty < \epsilon_{\text{fp32}} \approx 10^{-6}$$

In [ ]:
# Build a CNN model in PyTorch
class ConvClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model and export
model = ConvClassifier(num_classes=10)
model.eval()

dummy_input = torch.randn(1, 3, 32, 32)
torch.onnx.export(
    model, dummy_input, 'conv_classifier.onnx',
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17,
    do_constant_folding=True,
)

# Verify numerical equivalence
session = ort.InferenceSession('conv_classifier.onnx', providers=['CPUExecutionProvider'])

n_test = 100
max_diffs = []
for _ in range(n_test):
    x = np.random.randn(1, 3, 32, 32).astype(np.float32)
    
    with torch.no_grad():
        pt_out = model(torch.from_numpy(x)).numpy()
    ort_out = session.run(None, {'image': x})[0]
    
    max_diffs.append(np.abs(pt_out - ort_out).max())

print(f"Numerical Equivalence Verification ({n_test} samples):")
print(f"  Max difference: {max(max_diffs):.2e}")
print(f"  Mean difference: {np.mean(max_diffs):.2e}")
print(f"  All within 1e-5: {'✓ PASS' if max(max_diffs) < 1e-5 else '✗ FAIL'}")

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: {total_params:,} parameters")
print(f"File size: {os.path.getsize('conv_classifier.onnx') / 1024:.1f} KB")

<a id='section-3'></a>
## Section 3: Performance Comparison — Multiple Providers

ONNX Runtime applies graph optimizations at session creation. We measure the actual performance difference between PyTorch eager mode and ORT with various optimization levels:

- **ORT_DISABLE_ALL**: No graph optimizations
- **ORT_ENABLE_BASIC**: Basic optimizations (constant folding)
- **ORT_ENABLE_EXTENDED**: Extended optimizations (fusion)
- **ORT_ENABLE_ALL**: All optimizations

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

def benchmark(fn, n_warmup=50, n_iter=500):
    for _ in range(n_warmup):
        fn()
    times = []
    for _ in range(n_iter):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return np.array(times) * 1000  # ms

x_input = np.random.randn(8, 3, 32, 32).astype(np.float32)
x_torch = torch.from_numpy(x_input)

# PyTorch baseline
model.eval()
pt_times = benchmark(lambda: model(x_torch).detach())

# ORT with different optimization levels
opt_levels = {
    'Disabled': ort.GraphOptimizationLevel.ORT_DISABLE_ALL,
    'Basic': ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
    'Extended': ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
    'All': ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
}

ort_results = {}
for name, level in opt_levels.items():
    opts = ort.SessionOptions()
    opts.graph_optimization_level = level
    sess = ort.InferenceSession('conv_classifier.onnx', opts, providers=['CPUExecutionProvider'])
    times = benchmark(lambda: sess.run(None, {'image': x_input}))
    ort_results[name] = times

# Results
print(f"Performance Benchmark (batch_size=8, input=3×32×32)")
print("=" * 60)
print(f"{'Engine':<25} {'Mean (ms)':<12} {'P50 (ms)':<12} {'P99 (ms)':<12}")
print("─" * 60)
print(f"{'PyTorch (eager)':<25} {pt_times.mean():<12.3f} {np.percentile(pt_times, 50):<12.3f} {np.percentile(pt_times, 99):<12.3f}")
for name, times in ort_results.items():
    print(f"{'ORT (' + name + ')':<25} {times.mean():<12.3f} {np.percentile(times, 50):<12.3f} {np.percentile(times, 99):<12.3f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

all_data = [pt_times] + list(ort_results.values())
labels = ['PyTorch'] + [f'ORT ({k})' for k in ort_results.keys()]
colors = ['#FF6B6B'] + ['#4ECDC4', '#45B7D1', '#96CEB4', '#6BCB77']

bp = ax1.boxplot(all_data, labels=labels, patch_artist=True, medianprops=dict(color='black'))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Latency Distribution')
ax1.tick_params(axis='x', rotation=30)
ax1.grid(axis='y', alpha=0.3)

means = [d.mean() for d in all_data]
speedups = [means[0] / m for m in means]
ax2.bar(labels, speedups, color=colors, edgecolor='black', linewidth=0.5)
ax2.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
ax2.set_ylabel('Speedup vs PyTorch')
ax2.set_title('Speedup Factor')
ax2.tick_params(axis='x', rotation=30)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('PyTorch vs ONNX Runtime: Optimization Levels', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-4'></a>
## Section 4: Multi-Target Deployment from Single Model

One ONNX model can be deployed to multiple targets with different session configurations. This demonstrates the key portability benefit.

In [ ]:
import onnxruntime as ort

# Demonstrate multi-target configuration
model_path = 'conv_classifier.onnx'

# Configuration for different deployment targets
deploy_configs = {
    'Cloud Server (CPU, max throughput)': {
        'providers': ['CPUExecutionProvider'],
        'options': {
            'graph_optimization_level': ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
            'intra_op_num_threads': 4,
            'inter_op_num_threads': 2,
        }
    },
    'Edge Device (CPU, low latency)': {
        'providers': ['CPUExecutionProvider'],
        'options': {
            'graph_optimization_level': ort.GraphOptimizationLevel.ORT_ENABLE_ALL,
            'intra_op_num_threads': 1,
            'inter_op_num_threads': 1,
        }
    },
    'Mobile (CPU, minimal memory)': {
        'providers': ['CPUExecutionProvider'],
        'options': {
            'graph_optimization_level': ort.GraphOptimizationLevel.ORT_ENABLE_BASIC,
            'intra_op_num_threads': 1,
            'enable_mem_pattern': False,
        }
    },
}

print("Multi-Target Deployment Configurations")
print("=" * 60)

results = {}
for target_name, config in deploy_configs.items():
    opts = ort.SessionOptions()
    opts.graph_optimization_level = config['options']['graph_optimization_level']
    opts.intra_op_num_threads = config['options'].get('intra_op_num_threads', 0)
    opts.inter_op_num_threads = config['options'].get('inter_op_num_threads', 0)
    
    sess = ort.InferenceSession(model_path, opts, providers=config['providers'])
    
    # Benchmark
    x = np.random.randn(1, 3, 32, 32).astype(np.float32)
    times = benchmark(lambda: sess.run(None, {'image': x}), n_warmup=20, n_iter=200)
    
    results[target_name] = times
    print(f"\n{target_name}:")
    print(f"  Latency: {times.mean():.3f} ms (P50: {np.percentile(times, 50):.3f} ms)")
    print(f"  Throughput: {1000/times.mean():.0f} inferences/sec")

# Same model, different performance profiles!
print("\n" + "=" * 60)
print("Key insight: SAME .onnx file, DIFFERENT deployment profiles")

<a id='section-5'></a>
## Section 5: Graph Optimization Effects

Let's examine what optimizations ORT actually applies by saving the optimized model and comparing before/after.

In [ ]:
import onnx
from collections import Counter

# Save optimized model to inspect transformations
opts = ort.SessionOptions()
opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
opts.optimized_model_filepath = 'conv_classifier_optimized.onnx'
_ = ort.InferenceSession('conv_classifier.onnx', opts, providers=['CPUExecutionProvider'])

# Compare original vs optimized
original = onnx.load('conv_classifier.onnx')
optimized = onnx.load('conv_classifier_optimized.onnx')

orig_ops = Counter(n.op_type for n in original.graph.node)
opt_ops = Counter(n.op_type for n in optimized.graph.node)

print("Graph Optimization Analysis")
print("=" * 60)
print(f"\n{'Metric':<30} {'Original':<15} {'Optimized':<15}")
print("─" * 60)
print(f"{'Total nodes':<30} {len(original.graph.node):<15} {len(optimized.graph.node):<15}")
print(f"{'Initializers':<30} {len(original.graph.initializer):<15} {len(optimized.graph.initializer):<15}")
print(f"{'File size (bytes)':<30} {os.path.getsize('conv_classifier.onnx'):<15} {os.path.getsize('conv_classifier_optimized.onnx'):<15}")

print(f"\nOperator Changes:")
print(f"  {'Operator':<25} {'Before':<10} {'After':<10} {'Change'}")
print(f"  {'─'*55}")
all_ops = sorted(set(list(orig_ops.keys()) + list(opt_ops.keys())))
for op in all_ops:
    before = orig_ops.get(op, 0)
    after = opt_ops.get(op, 0)
    change = after - before
    if change != 0:
        sign = '+' if change > 0 else ''
        print(f"  {op:<25} {before:<10} {after:<10} {sign}{change}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Original ops
ops_orig = sorted(orig_ops.items(), key=lambda x: -x[1])[:10]
ax1.barh([x[0] for x in ops_orig], [x[1] for x in ops_orig], color='#FF9999')
ax1.set_xlabel('Count')
ax1.set_title('Original Model — Top Operators')
ax1.grid(axis='x', alpha=0.3)

# Optimized ops
ops_opt = sorted(opt_ops.items(), key=lambda x: -x[1])[:10]
ax2.barh([x[0] for x in ops_opt], [x[1] for x in ops_opt], color='#99FF99')
ax2.set_xlabel('Count')
ax2.set_title('Optimized Model — Top Operators')
ax2.grid(axis='x', alpha=0.3)

plt.suptitle('Graph Optimization: Operator Distribution Change', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-6'></a>
## Section 6: Model Size and Memory Analysis

ONNX enables precise analysis of where model size comes from — crucial for edge deployment where memory is constrained.

In [ ]:
import onnx
from onnx import numpy_helper
import matplotlib.pyplot as plt
import numpy as np

model = onnx.load('conv_classifier.onnx')

# Analyze parameter sizes
param_info = []
for init in model.graph.initializer:
    arr = numpy_helper.to_array(init)
    param_info.append({
        'name': init.name,
        'shape': list(arr.shape),
        'dtype': str(arr.dtype),
        'params': arr.size,
        'bytes': arr.nbytes,
    })

# Sort by size
param_info.sort(key=lambda x: -x['bytes'])

print("Model Memory Breakdown")
print("=" * 70)
print(f"{'Parameter':<35} {'Shape':<20} {'Size':<10} {'Bytes':<12}")
print("─" * 70)
total_bytes = 0
for p in param_info:
    total_bytes += p['bytes']
    print(f"{p['name']:<35} {str(p['shape']):<20} {p['params']:<10} {p['bytes']:<12}")
print("─" * 70)
print(f"{'TOTAL':<35} {'':20} {sum(p['params'] for p in param_info):<10} {total_bytes:<12}")

# Visualize size distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart of parameter distribution
sizes = [p['bytes'] for p in param_info[:8]]
labels = [p['name'][:20] for p in param_info[:8]]
if len(param_info) > 8:
    sizes.append(sum(p['bytes'] for p in param_info[8:]))
    labels.append('Other')

colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, autopct='%1.1f%%',
                                     colors=colors, textprops={'fontsize': 7})
ax1.set_title('Parameter Size Distribution', fontsize=12)

# Bar chart comparing model size under different precisions
precisions = ['FP32\n(original)', 'FP16', 'INT8\n(quantized)']
size_multipliers = [1.0, 0.5, 0.25]
model_sizes = [total_bytes * m / 1024 for m in size_multipliers]

bars = ax2.bar(precisions, model_sizes, color=['#FF6B6B', '#FFD93D', '#4ECDC4'],
               edgecolor='black', linewidth=1)
ax2.set_ylabel('Model Size (KB)')
ax2.set_title('Model Size by Precision')
for bar, size in zip(bars, model_sizes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{size:.1f} KB', ha='center', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Model Size Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-7'></a>
## Section 7: Batch Processing and Throughput

ONNX models with dynamic axes support variable batch sizes. We analyze the throughput-latency trade-off:

$$\text{Throughput} = \frac{\text{batch\_size}}{\text{latency(batch\_size)}}$$

The relationship is typically sub-linear due to memory bandwidth saturation.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt

session = ort.InferenceSession('conv_classifier.onnx', providers=['CPUExecutionProvider'])

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
latencies = []
throughputs = []

for bs in batch_sizes:
    x = np.random.randn(bs, 3, 32, 32).astype(np.float32)
    
    # Warmup
    for _ in range(20):
        session.run(None, {'image': x})
    
    # Measure
    times = []
    for _ in range(100):
        start = time.perf_counter()
        session.run(None, {'image': x})
        times.append(time.perf_counter() - start)
    
    avg_latency = np.mean(times) * 1000  # ms
    avg_throughput = bs / np.mean(times)  # samples/sec
    latencies.append(avg_latency)
    throughputs.append(avg_throughput)

print(f"Batch Size vs Performance")
print("=" * 60)
print(f"{'Batch':<8} {'Latency (ms)':<15} {'Throughput (samples/s)':<25} {'Per-sample (ms)'}")
print("─" * 60)
for bs, lat, thr in zip(batch_sizes, latencies, throughputs):
    print(f"{bs:<8} {lat:<15.3f} {thr:<25.1f} {lat/bs:.3f}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

ax1.plot(batch_sizes, latencies, 'o-', color='#FF6B6B', linewidth=2, markersize=8)
ax1.set_xlabel('Batch Size')
ax1.set_ylabel('Total Latency (ms)')
ax1.set_title('Latency vs Batch Size')
ax1.set_xscale('log', base=2)
ax1.grid(alpha=0.3)

ax2.plot(batch_sizes, throughputs, 's-', color='#4ECDC4', linewidth=2, markersize=8)
ax2.set_xlabel('Batch Size')
ax2.set_ylabel('Throughput (samples/sec)')
ax2.set_title('Throughput vs Batch Size')
ax2.set_xscale('log', base=2)
ax2.grid(alpha=0.3)

per_sample = [l/bs for l, bs in zip(latencies, batch_sizes)]
ax3.plot(batch_sizes, per_sample, 'D-', color='#45B7D1', linewidth=2, markersize=8)
ax3.set_xlabel('Batch Size')
ax3.set_ylabel('Per-sample Latency (ms)')
ax3.set_title('Amortization Effect')
ax3.set_xscale('log', base=2)
ax3.grid(alpha=0.3)

plt.suptitle('Batch Processing Analysis: Latency-Throughput Trade-off', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Building a Production Inference Pipeline

Let's build a complete inference pipeline that demonstrates production best practices: input validation, session pooling, error handling, and output post-processing.

In [ ]:
import numpy as np
import onnxruntime as ort
import time

class ONNXInferencePipeline:
    """Production-ready ONNX inference pipeline."""
    
    def __init__(self, model_path, providers=None, num_threads=4):
        opts = ort.SessionOptions()
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        opts.intra_op_num_threads = num_threads
        opts.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        
        if providers is None:
            providers = ['CPUExecutionProvider']
        
        self.session = ort.InferenceSession(model_path, opts, providers=providers)
        self.input_info = self.session.get_inputs()[0]
        self.output_info = self.session.get_outputs()[0]
        self._call_count = 0
        self._total_time = 0
    
    def validate_input(self, x):
        """Validate input shape and type."""
        expected_shape = self.input_info.shape
        if x.dtype != np.float32:
            x = x.astype(np.float32)
        # Check static dimensions
        for i, (actual, expected) in enumerate(zip(x.shape, expected_shape)):
            if isinstance(expected, int) and actual != expected:
                raise ValueError(f"Dimension {i}: expected {expected}, got {actual}")
        return x
    
    def predict(self, x, return_probs=True):
        """Run inference with pre/post processing."""
        x = self.validate_input(x)
        
        start = time.perf_counter()
        logits = self.session.run(None, {self.input_info.name: x})[0]
        elapsed = time.perf_counter() - start
        
        self._call_count += 1
        self._total_time += elapsed
        
        if return_probs:
            exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
            probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
            return probs
        return logits
    
    def predict_class(self, x):
        """Return predicted class indices."""
        probs = self.predict(x)
        return np.argmax(probs, axis=1)
    
    @property
    def stats(self):
        if self._call_count == 0:
            return {'calls': 0}
        return {
            'calls': self._call_count,
            'total_time_ms': self._total_time * 1000,
            'avg_latency_ms': self._total_time / self._call_count * 1000,
            'throughput_per_sec': self._call_count / self._total_time,
        }

# Use the pipeline
pipeline = ONNXInferencePipeline('conv_classifier.onnx')

# Simulate production workload
n_requests = 1000
batch_sizes_used = np.random.choice([1, 2, 4, 8], size=n_requests, p=[0.5, 0.25, 0.15, 0.1])

for bs in batch_sizes_used:
    x = np.random.randn(bs, 3, 32, 32).astype(np.float32)
    predictions = pipeline.predict_class(x)

stats = pipeline.stats
print("Production Pipeline Statistics")
print("=" * 50)
print(f"  Total requests:      {stats['calls']:,}")
print(f"  Total time:          {stats['total_time_ms']:.1f} ms")
print(f"  Avg latency:         {stats['avg_latency_ms']:.3f} ms")
print(f"  Throughput:          {stats['throughput_per_sec']:.0f} requests/sec")
print(f"  Input spec:          {pipeline.input_info.name} {pipeline.input_info.shape}")
print(f"  Output spec:         {pipeline.output_info.name} {pipeline.output_info.shape}")

<a id='section-9'></a>
## Section 9: Exercises

### Exercise 1: Latency Budget Analysis
Given a service SLA of 50ms P99 latency, determine the maximum batch size for our CNN model.

### Exercise 2: Model Ensemble
Create two different ONNX models (e.g., a linear model and the CNN) and implement an ensemble that averages their predictions.

### Exercise 3: Thread Scaling
Benchmark the CNN model with 1, 2, 4, and 8 `intra_op_num_threads` and plot the speedup curve.

### Exercise 4: Memory Profiling
Use `tracemalloc` to measure peak memory usage during ORT inference vs PyTorch inference.

In [ ]:
# Cleanup
import os
for f in ['conv_classifier.onnx', 'conv_classifier_optimized.onnx']:
    if os.path.exists(f):
        os.remove(f)

print("Notebook complete! Key takeaways:")
print("  1. ONNX Runtime provides measurable speedups over PyTorch eager mode")
print("  2. Graph optimizations (fusion, folding) reduce node count significantly")
print("  3. Single ONNX model serves multiple deployment targets")
print("  4. Batch processing enables throughput-latency trade-off tuning")
print("  5. Production pipelines need validation, error handling, and monitoring")

---

**Next:** [ONNX Ecosystem Overview — Deep Dive](../03_ONNX_Ecosystem_Overview/ONNX_Ecosystem_Overview_Deep_Dive.ipynb)